# Arquitectura y composición interna de los microcontroladores · Laboratorio en Colab

**Circuitos eléctricos y microcontroladores · PIC16F877 / PIC16F877A**

Este cuaderno acompaña la presentación. Todo corre en Python, sin instalar nada:

1. Un **ensamblador** (subconjunto de MPASM) y un **emulador** del núcleo PIC16 escritos para la clase.
2. Los **cuatro ejemplos** de la presentación: se ensamblan aquí y generan el `.hex` que se carga en Proteus o en el PIC real.
3. **Demostraciones de arquitectura**: Von Neumann frente a Harvard, CISC frente a RISC, segmentación (pipeline) y codificación de instrucciones de 14 bits.
4. **Cálculos de diseño**: retardos, resistencias de LED, ADC, PWM y velocidad de la USART.
5. **Simulación** de cada ejemplo con gráficas y **ejercicios propuestos**.

> Ejecute las celdas en orden (*Entorno de ejecución → Ejecutar todas*). Las celdas que empiezan con `%%writefile` solo guardan archivos.

## 1. Herramientas: ensamblador y emulador
El ensamblador acepta la sintaxis de MPASM usada en clase (`LIST`, `#include`, `__CONFIG`, `ORG`, `EQU`, `CBLOCK`, `BANKSEL`, `DT`). El emulador ejecuta las 35 instrucciones con su temporización real (1 ciclo de instrucción = 4 periodos del cristal) y modela puertos, ADC, Timer2/PWM y USART.

In [ ]:
%%writefile picasm.py
"""picasm: ensamblador mínimo compatible con la sintaxis MPASM para PIC16F877/877A (núcleo de 14 bits).
Soporta: LIST, #include <p16f877(a).inc>, __CONFIG, ORG, EQU, CBLOCK/ENDC, DT, BANKSEL, END, etiquetas,
números 0x.., h'..', b'..', d'..', .123, 'A', y expresiones con + - * / | & << >> y paréntesis.
Genera Intel HEX (INHX8M) listo para MPLAB, Proteus o un programador."""
import re

SFR = {
    'INDF': 0x00, 'TMR0': 0x01, 'PCL': 0x02, 'STATUS': 0x03, 'FSR': 0x04, 'PORTA': 0x05, 'PORTB': 0x06, 'PORTC': 0x07,
    'PORTD': 0x08, 'PORTE': 0x09, 'PCLATH': 0x0A, 'INTCON': 0x0B, 'PIR1': 0x0C, 'PIR2': 0x0D, 'TMR1L': 0x0E, 'TMR1H': 0x0F,
    'T1CON': 0x10, 'TMR2': 0x11, 'T2CON': 0x12, 'SSPBUF': 0x13, 'SSPCON': 0x14, 'CCPR1L': 0x15, 'CCPR1H': 0x16,
    'CCP1CON': 0x17, 'RCSTA': 0x18, 'TXREG': 0x19, 'RCREG': 0x1A, 'CCPR2L': 0x1B, 'CCPR2H': 0x1C, 'CCP2CON': 0x1D,
    'ADRESH': 0x1E, 'ADCON0': 0x1F, 'OPTION_REG': 0x81, 'TRISA': 0x85, 'TRISB': 0x86, 'TRISC': 0x87, 'TRISD': 0x88,
    'TRISE': 0x89, 'PIE1': 0x8C, 'PIE2': 0x8D, 'PCON': 0x8E, 'SSPCON2': 0x91, 'PR2': 0x92, 'SSPADD': 0x93, 'SSPSTAT': 0x94,
    'TXSTA': 0x98, 'SPBRG': 0x99, 'CMCON': 0x9C, 'CVRCON': 0x9D, 'ADRESL': 0x9E, 'ADCON1': 0x9F, 'EEDATA': 0x10C,
    'EEADR': 0x10D, 'EEDATH': 0x10E, 'EEADRH': 0x10F, 'EECON1': 0x18C, 'EECON2': 0x18D,
}
BITS = {
    'C': 0, 'DC': 1, 'Z': 2, 'NOT_PD': 3, 'NOT_TO': 4, 'RP0': 5, 'RP1': 6, 'IRP': 7,
    'GO': 2, 'GO_DONE': 2, 'NOT_DONE': 2, 'ADON': 0, 'ADFM': 7, 'CHS0': 3, 'CHS1': 4, 'CHS2': 5, 'ADCS0': 6, 'ADCS1': 7,
    'TMR2ON': 2, 'T2CKPS0': 0, 'T2CKPS1': 1, 'TXIF': 4, 'RCIF': 5, 'ADIF': 6, 'TMR2IF': 1, 'TMR1IF': 0, 'CCP1IF': 2,
    'TXEN': 5, 'BRGH': 2, 'TRMT': 1, 'SYNC': 4, 'SPEN': 7, 'CREN': 4, 'OERR': 1, 'FERR': 2,
    'GIE': 7, 'PEIE': 6, 'T0IE': 5, 'INTE': 4, 'RBIE': 3, 'T0IF': 2, 'INTF': 1, 'RBIF': 0,
    'NOT_RBPU': 7, 'INTEDG': 6, 'T0CS': 5, 'T0SE': 4, 'PSA': 3, 'PS2': 2, 'PS1': 1, 'PS0': 0, 'W': 0, 'F': 1,
}
for p in 'ABCDE':
    for i in range(8):
        BITS[f'R{p}{i}'] = i
CONFIG = {  # valores del archivo p16f877a.inc (se combinan con &)
    '_CP_OFF': 0x3FFF, '_CP_ALL': 0x1FFF, '_DEBUG_OFF': 0x3FFF, '_DEBUG_ON': 0x37FF, '_WRT_OFF': 0x3FFF, '_CPD_OFF': 0x3FFF,
    '_CPD_ON': 0x3EFF, '_LVP_OFF': 0x3F7F, '_LVP_ON': 0x3FFF, '_BODEN_OFF': 0x3FBF, '_BOREN_OFF': 0x3FBF, '_BODEN_ON': 0x3FFF,
    '_PWRTE_ON': 0x3FF7, '_PWRTE_OFF': 0x3FFF, '_WDT_OFF': 0x3FFB, '_WDT_ON': 0x3FFF, '_WDTE_OFF': 0x3FFB,
    '_LP_OSC': 0x3FFC, '_XT_OSC': 0x3FFD, '_HS_OSC': 0x3FFE, '_RC_OSC': 0x3FFF,
}
BYTE_OPS = {'ADDWF': 0x0700, 'ANDWF': 0x0500, 'COMF': 0x0900, 'DECF': 0x0300, 'DECFSZ': 0x0B00, 'INCF': 0x0A00,
            'INCFSZ': 0x0F00, 'IORWF': 0x0400, 'MOVF': 0x0800, 'RLF': 0x0D00, 'RRF': 0x0C00, 'SUBWF': 0x0200,
            'SWAPF': 0x0E00, 'XORWF': 0x0600}
BIT_OPS = {'BCF': 0x1000, 'BSF': 0x1400, 'BTFSC': 0x1800, 'BTFSS': 0x1C00}
LIT_OPS = {'ADDLW': 0x3E00, 'ANDLW': 0x3900, 'IORLW': 0x3800, 'MOVLW': 0x3000, 'RETLW': 0x3400, 'SUBLW': 0x3C00, 'XORLW': 0x3A00}
NOARG = {'NOP': 0x0000, 'CLRW': 0x0100, 'CLRWDT': 0x0064, 'RETFIE': 0x0009, 'RETURN': 0x0008, 'SLEEP': 0x0063}


class ErrorASM(Exception):
    pass


class Ensamblador:
    def __init__(self, fuente):
        self.fuente = fuente
        self.sim = {}

    # ---------- expresiones ----------
    def numero(self, t):
        t = t.strip()
        m = re.fullmatch(r"([hHbBdDoO])'([^']*)'", t)
        if m:
            base = {'h': 16, 'b': 2, 'd': 10, 'o': 8}[m.group(1).lower()]
            return int(m.group(2), base)
        if re.fullmatch(r"'(.)'", t): return ord(t[1])
        if t.lower().startswith('0x'): return int(t, 16)
        if t.startswith('.'): return int(t[1:])
        if re.fullmatch(r'[0-9A-Fa-f]+[hH]', t): return int(t[:-1], 16)
        if re.fullmatch(r'[01]+[bB]', t): return int(t[:-1], 2)
        if re.fullmatch(r'\d+', t): return int(t, 16 if self.radix == 16 else 10)
        raise ValueError

    def valor(self, expr, pasada):
        expr = expr.strip()
        toks = re.findall(r"[hHbBdDoO]'[^']*'|'.'|0[xX][0-9A-Fa-f]+|\.\d+|[A-Za-z_][A-Za-z0-9_]*|\d[0-9A-Fa-f]*[hHbB]?|<<|>>|[-+*/|&()~^]", expr)
        py = []
        for t in toks:
            if re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', t) and not re.fullmatch(r"[hHbBdDoO]", t):
                u = t.upper()
                if u in self.sim: py.append(str(self.sim[u]))
                elif u in SFR: py.append(str(SFR[u]))
                elif u in BITS: py.append(str(BITS[u]))
                elif u in CONFIG: py.append(str(CONFIG[u]))
                elif pasada == 1: py.append('0')
                else: raise ErrorASM(f'símbolo no definido: {t}')
            elif t in ('<<', '>>', '+', '-', '*', '|', '&', '(', ')', '~', '^'):
                py.append(t)
            elif t == '/':
                py.append('//')
            else:
                try: py.append(str(self.numero(t)))
                except ValueError: raise ErrorASM(f'número no válido: {t}')
        return int(eval(' '.join(py), {'__builtins__': {}}))

    # ---------- utilidades ----------
    @staticmethod
    def partir(ops):
        out, prof, cur, com = [], 0, '', False
        for ch in ops:
            if ch == "'" : com = not com
            if ch == ',' and prof == 0 and not com: out.append(cur); cur = ''; continue
            if ch == '(': prof += 1
            if ch == ')': prof -= 1
            cur += ch
        if cur.strip(): out.append(cur)
        return [o.strip() for o in out]

    def ensamblar(self):
        for pasada in (1, 2):
            self.radix = 16
            self.pc = 0
            self.mem = {}
            self.config = None
            self.lista = []
            cblock = None
            for nlin, linea in enumerate(self.fuente.split('\n'), 1):
                cod = linea.split(';')[0].rstrip()
                if not cod.strip():
                    continue
                try:
                    if cblock is not None:
                        if cod.strip().upper() == 'ENDC': cblock = None; continue
                        for nombre in cod.replace(',', ' ').split():
                            if pasada == 1: self.sim[nombre.upper()] = cblock
                            cblock += 1
                        continue
                    etiqueta = None
                    if not cod[0].isspace():
                        partes = cod.split(None, 1)
                        etiqueta = partes[0].rstrip(':')
                        cod = ' ' + (partes[1] if len(partes) > 1 else '')
                    partes = cod.strip().split(None, 1)
                    op = partes[0].upper() if partes else ''
                    args = partes[1] if len(partes) > 1 else ''
                    if etiqueta and etiqueta.upper().startswith('#'):
                        op, args, etiqueta = etiqueta.upper(), cod.strip(), None
                    if op == 'EQU':
                        if pasada == 1: self.sim[etiqueta.upper()] = self.valor(args, pasada)
                        continue
                    if etiqueta:
                        if etiqueta.upper() in ('LIST', 'END'):
                            op, args = etiqueta.upper(), cod.strip()
                        else:
                            if pasada == 1:
                                if etiqueta.upper() in self.sim: raise ErrorASM(f'etiqueta repetida: {etiqueta}')
                                self.sim[etiqueta.upper()] = self.pc
                    if not op: continue
                    if op in ('LIST', 'PROCESSOR', '#INCLUDE', 'ERRORLEVEL', 'RADIX', 'TITLE'):
                        if op == 'LIST':
                            m = re.search(r'[rR]\s*=\s*(\w+)', args)
                            if m: self.radix = 10 if m.group(1).upper().startswith('DEC') else 16
                        if op == 'RADIX': self.radix = 10 if args.strip().upper().startswith('DEC') else 16
                        continue
                    if op == 'END': break
                    if op == '__CONFIG':
                        a = self.partir(args)
                        self.config = self.valor(a[-1], pasada) & 0x3FFF
                        continue
                    if op == 'ORG': self.pc = self.valor(args, pasada); continue
                    if op == 'CBLOCK': cblock = self.valor(args, pasada); continue
                    if op == 'DT':
                        for a in self.partir(args):
                            if a.startswith('"'):
                                for ch in a.strip('"'): self.emitir(0x3400 | ord(ch), f'retlw {ord(ch):#04x}', nlin)
                            else:
                                self.emitir(0x3400 | (self.valor(a, pasada) & 0xFF), f'retlw {a}', nlin)
                        continue
                    if op == 'BANKSEL':
                        dirb = self.valor(args, pasada)
                        banco = (dirb >> 7) & 3
                        self.emitir((0x1400 if banco & 1 else 0x1000) | (5 << 7) | 3, 'bsf/bcf STATUS,RP0', nlin)
                        self.emitir((0x1400 if banco & 2 else 0x1000) | (6 << 7) | 3, 'bsf/bcf STATUS,RP1', nlin)
                        continue
                    a = self.partir(args)
                    if op in NOARG: w = NOARG[op]
                    elif op in BYTE_OPS:
                        f = self.valor(a[0], pasada) & 0x7F
                        d = 1 if len(a) < 2 else self.valor(a[1], pasada) & 1
                        w = BYTE_OPS[op] | (d << 7) | f
                    elif op == 'MOVWF': w = 0x0080 | (self.valor(a[0], pasada) & 0x7F)
                    elif op == 'CLRF': w = 0x0180 | (self.valor(a[0], pasada) & 0x7F)
                    elif op in BIT_OPS:
                        f = self.valor(a[0], pasada) & 0x7F; b = self.valor(a[1], pasada) & 7
                        w = BIT_OPS[op] | (b << 7) | f
                    elif op in LIT_OPS: w = LIT_OPS[op] | (self.valor(a[0], pasada) & 0xFF)
                    elif op in ('CALL', 'GOTO'):
                        k = self.valor(a[0], pasada)
                        w = (0x2000 if op == 'CALL' else 0x2800) | (k & 0x7FF)
                    else:
                        raise ErrorASM(f'instrucción desconocida: {op}')
                    self.emitir(w, linea.strip(), nlin)
                except ErrorASM as e:
                    raise ErrorASM(f'línea {nlin}: {e}  →  {linea.strip()}')
        return self

    def emitir(self, w, txt, nlin):
        self.mem[self.pc] = w & 0x3FFF
        self.lista.append((self.pc, w & 0x3FFF, txt))
        self.pc += 1

    # ---------- Intel HEX ----------
    def hex(self):
        datos = {}
        for a, w in self.mem.items():
            datos[2 * a] = w & 0xFF; datos[2 * a + 1] = w >> 8
        if self.config is not None:
            datos[0x400E] = self.config & 0xFF; datos[0x400F] = self.config >> 8
        lineas = [':020000040000FA']
        dirs = sorted(datos)
        i = 0
        while i < len(dirs):
            ini = dirs[i]; bloque = [datos[ini]]; j = i + 1
            while j < len(dirs) and dirs[j] == ini + len(bloque) and len(bloque) < 16 and (ini & ~0xF) == (dirs[j] & ~0xF):
                bloque.append(datos[dirs[j]]); j += 1
            reg = [len(bloque), (ini >> 8) & 0xFF, ini & 0xFF, 0] + bloque
            reg.append((-sum(reg)) & 0xFF)
            lineas.append(':' + ''.join(f'{b:02X}' for b in reg))
            i = j
        lineas.append(':00000001FF')
        return '\n'.join(lineas) + '\n'


def leer_hex(texto):
    """Lee un Intel HEX de PIC y devuelve (memoria de programa {dir: palabra}, palabra de configuración)."""
    datos, base = {}, 0
    for l in texto.split():
        if not l.startswith(':'): continue
        b = bytes.fromhex(l[1:])
        if sum(b) & 0xFF: raise ValueError('suma de verificación incorrecta: ' + l)
        n, dirc, tipo = b[0], (b[1] << 8) | b[2], b[3]
        if tipo == 0:
            for k in range(n): datos[base + dirc + k] = b[4 + k]
        elif tipo == 4: base = ((b[4] << 8) | b[5]) << 16
        elif tipo == 1: break
    prog, conf = {}, None
    for a in sorted(datos):
        if a % 2 == 0 and a + 1 in datos:
            w = datos[a] | (datos[a + 1] << 8)
            if a // 2 == 0x2007: conf = w
            elif a // 2 < 0x2000: prog[a // 2] = w
    return prog, conf


def desensamblar(w):
    for k, v in NOARG.items():
        if w == v: return k.lower()
    if w & 0x3F9F == 0 and w != 0: return 'nop'
    top = w & 0x3000
    if top == 0x0000:
        f, d = w & 0x7F, (w >> 7) & 1
        o = w & 0x0F00
        if o == 0x0000: return f'movwf 0x{f:02X}'
        if o == 0x0100: return f'clrf 0x{f:02X}'
        for k, v in BYTE_OPS.items():
            if v == o: return f'{k.lower()} 0x{f:02X},{"F" if d else "W"}'
    if top == 0x1000:
        o = w & 0x1C00; b = (w >> 7) & 7; f = w & 0x7F
        for k, v in BIT_OPS.items():
            if v == o: return f'{k.lower()} 0x{f:02X},{b}'
    if top == 0x2000:
        return ('goto' if w & 0x0800 else 'call') + f' 0x{w & 0x7FF:03X}'
    k = w & 0xFF
    for nm, v in (('movlw', 0x3000), ('retlw', 0x3400), ('iorlw', 0x3800), ('andlw', 0x3900), ('xorlw', 0x3A00), ('sublw', 0x3C00), ('addlw', 0x3E00)):
        mask = 0x3F00 if nm in ('iorlw', 'andlw', 'xorlw') else (0x3C00 if nm in ('movlw', 'retlw') else 0x3E00)
        if w & mask == v: return f'{nm} 0x{k:02X}'
    return f'dw 0x{w:04X}'

In [ ]:
%%writefile pic16sim.py
"""pic16sim: emulador didáctico del núcleo PIC16F877/877A (35 instrucciones) con puertos, ADC, TMR2/PWM y USART.
No modela interrupciones, TMR0/TMR1, MSSP ni EEPROM: suficiente para verificar los ejemplos de la clase."""

class PIC16F877:
    COMUN = {0x00, 0x02, 0x03, 0x04, 0x0A, 0x0B}  # registros reflejados en los 4 bancos

    def __init__(self, prog, fosc=4e6):
        self.prog = [0x3FFF] * 8192
        for a, w in prog.items(): self.prog[a] = w
        self.fosc = fosc
        self.ram = [0] * 512
        self.w = 0; self.pc = 0; self.pila = []; self.ciclos = 0
        self.r(0x03, 0x18)
        for t in (0x85, 0x86, 0x87, 0x88, 0x89): self.ram[t] = 0xFF
        self.ram[0x89] = 0x07; self.ram[0x81] = 0xFF; self.ram[0x92] = 0xFF
        self.ram[0x98] = 0x02  # TXSTA.TRMT = 1
        self.ram[0x0C] = 0x10  # PIR1.TXIF = 1
        self.entradas = {p: 0 for p in 'ABCDE'}   # nivel externo de los pines configurados como entrada
        self.analog = [0.0] * 8                   # voltajes en AN0..AN7
        self.adc_fin = None
        self.tsr_fin = None; self.tsr = None; self.txreg_lleno = False
        self.tx = []                              # caracteres transmitidos por la USART
        self.escrituras = []                      # (ciclo, registro, valor) para depurar

    # ---------- mapa de memoria ----------
    def dirf(self, f):
        if f == 0: f = self.ram[0x04] | ((self.ram[0x03] & 0x80) << 1)   # INDF → FSR (IRP)
        elif True:
            f |= (self.ram[0x03] & 0x60) << 2
        b = f & 0x7F
        if b in self.COMUN: return b
        if b >= 0x70: return b
        if f in (0x101, 0x106, 0x181, 0x186): return f & 0xFF
        return f

    def r(self, a, v=None):
        if v is None: return self.ram[a]
        self.ram[a] = v & 0xFF

    PUERTOS = {0x05: 'A', 0x06: 'B', 0x07: 'C', 0x08: 'D', 0x09: 'E'}

    def leer(self, a):
        if a in self.PUERTOS:
            p = self.PUERTOS[a]; tris = self.ram[a + 0x80]
            return ((self.ram[a] & ~tris) | (self.entradas[p] & tris)) & 0xFF
        if a == 0x02: return self.pc & 0xFF
        if a == 0x1A:  # RCREG
            self.ram[0x0C] &= ~0x20
        return self.ram[a]

    def escribir(self, a, v):
        v &= 0xFF
        if a == 0x02:
            self.ram[2] = v; self.pc = ((self.ram[0x0A] & 0x1F) << 8) | v; self.ciclos += 1
            return True
        if a == 0x03: v = (self.ram[3] & 0x18) | (v & 0xE7)
        self.ram[a] = v
        self.escrituras.append((self.ciclos, a, v))
        if a == 0x1F and (v & 0x05) == 0x05 and self.adc_fin is None:   # GO con ADON
            self.adc_fin = self.ciclos + 25
        if a == 0x19:  # TXREG
            if self.tsr is None: self.tsr = v; self.tsr_fin = self.ciclos + self.ciclos_char()
            else: self.txreg_lleno = True; self.ram[0x0C] &= ~0x10
        return False

    def ciclos_char(self):
        brgh = (self.ram[0x98] >> 2) & 1
        baud = self.fosc / ((16 if brgh else 64) * (self.ram[0x99] + 1))
        return int(10 * (self.fosc / 4) / baud)

    def perifericos(self):
        if self.adc_fin is not None and self.ciclos >= self.adc_fin:
            ch = (self.ram[0x1F] >> 3) & 7
            res = max(0, min(1023, round(self.analog[ch] / 5.0 * 1023)))
            if self.ram[0x9F] & 0x80: self.ram[0x1E] = res >> 8; self.ram[0x9E] = res & 0xFF
            else: self.ram[0x1E] = res >> 2; self.ram[0x9E] = (res & 3) << 6
            self.ram[0x1F] &= ~0x04; self.ram[0x0C] |= 0x40; self.adc_fin = None
        if self.tsr is not None:
            self.ram[0x98] &= ~0x02
            if self.ciclos >= self.tsr_fin:
                self.tx.append(self.tsr); self.tsr = None
                if self.txreg_lleno:
                    self.txreg_lleno = False; self.tsr = self.ram[0x19]; self.tsr_fin = self.ciclos + self.ciclos_char()
                self.ram[0x0C] |= 0x10
                if self.tsr is None: self.ram[0x98] |= 0x02

    def recibir(self, ch):
        self.ram[0x1A] = ch & 0xFF; self.ram[0x0C] |= 0x20

    def pwm(self, n=1):
        ccpcon, ccprl, pin = (0x17, 0x15, 2) if n == 1 else (0x1D, 0x1B, 1)
        t2 = self.ram[0x12]
        if (self.ram[ccpcon] & 0x0C) != 0x0C or not (t2 & 0x04): return None
        pres = (1, 4, 16, 16)[t2 & 3]
        per = (self.ram[0x92] + 1) * 4 * pres / self.fosc
        dc = ((self.ram[ccprl] << 2) | ((self.ram[ccpcon] >> 4) & 3)) * pres / self.fosc
        return {'f': 1 / per, 'D': min(1.0, dc / per), 'salida': not (self.ram[0x87] >> pin) & 1}

    # ---------- núcleo ----------
    def flags(self, z=None, c=None, dc=None):
        s = self.ram[3]
        if z is not None: s = (s | 4) if z else (s & ~4)
        if c is not None: s = (s | 1) if c else (s & ~1)
        if dc is not None: s = (s | 2) if dc else (s & ~2)
        self.ram[3] = s & 0xFF

    def paso(self):
        op = self.prog[self.pc]; pc0 = self.pc
        self.pc = (self.pc + 1) & 0x1FFF; self.ciclos += 1
        top = op & 0x3000
        if top == 0x0000:
            f = op & 0x7F; d = (op >> 7) & 1; o = op & 0x0F00
            if o == 0x0000:
                if op & 0x80: self.escribir(self.dirf(f), self.w)
                elif op == 0x0008: self.pc = self.pila.pop() if self.pila else 0; self.ciclos += 1
                elif op == 0x0009: self.pc = self.pila.pop() if self.pila else 0; self.ram[0x0B] |= 0x80; self.ciclos += 1
                elif op == 0x0063: raise StopIteration('SLEEP')
                return
            if o == 0x0100:
                if op & 0x80: self.escribir(self.dirf(f), 0); self.flags(z=True)
                else: self.w = 0; self.flags(z=True)
                return
            a = self.dirf(f); v = self.leer(a); skip = False
            if o == 0x0700: r = v + self.w; self.flags(z=(r & 0xFF) == 0, c=r > 255, dc=((v & 15) + (self.w & 15)) > 15)
            elif o == 0x0500: r = v & self.w; self.flags(z=(r & 0xFF) == 0)
            elif o == 0x0900: r = ~v & 0xFF; self.flags(z=r == 0)
            elif o == 0x0300: r = (v - 1) & 0xFF; self.flags(z=r == 0)
            elif o == 0x0B00: r = (v - 1) & 0xFF; skip = r == 0
            elif o == 0x0A00: r = (v + 1) & 0xFF; self.flags(z=r == 0)
            elif o == 0x0F00: r = (v + 1) & 0xFF; skip = r == 0
            elif o == 0x0400: r = v | self.w; self.flags(z=(r & 0xFF) == 0)
            elif o == 0x0800: r = v; self.flags(z=r == 0)
            elif o == 0x0D00: c = self.ram[3] & 1; r = ((v << 1) | c) & 0xFF; self.flags(c=bool(v & 0x80))
            elif o == 0x0C00: c = self.ram[3] & 1; r = (v >> 1) | (c << 7); self.flags(c=bool(v & 1))
            elif o == 0x0200: r = (v - self.w) & 0xFF; self.flags(z=r == 0, c=v >= self.w, dc=(v & 15) >= (self.w & 15))
            elif o == 0x0E00: r = ((v << 4) | (v >> 4)) & 0xFF
            elif o == 0x0600: r = v ^ self.w; self.flags(z=(r & 0xFF) == 0)
            else: raise ValueError(f'opcode {op:04X}')
            r &= 0xFF
            if d: self.escribir(a, r)
            else: self.w = r
            if skip: self.pc = (self.pc + 1) & 0x1FFF; self.ciclos += 1
            return
        if top == 0x1000:
            f = op & 0x7F; b = (op >> 7) & 7; o = op & 0x0C00; a = self.dirf(f)
            if o == 0x0000: self.escribir(a, self.ram[a] & ~(1 << b))
            elif o == 0x0400: self.escribir(a, self.ram[a] | (1 << b))
            else:
                bit = (self.leer(a) >> b) & 1
                if (o == 0x0800 and bit == 0) or (o == 0x0C00 and bit == 1):
                    self.pc = (self.pc + 1) & 0x1FFF; self.ciclos += 1
            return
        if top == 0x2000:
            k = op & 0x7FF; dest = ((self.ram[0x0A] & 0x18) << 8) | k
            if not op & 0x0800:
                if len(self.pila) == 8: self.pila.pop(0)
                self.pila.append(self.pc)
            self.pc = dest; self.ciclos += 1
            return
        k = op & 0xFF
        if op & 0x3C00 == 0x3000: self.w = k
        elif op & 0x3C00 == 0x3400: self.w = k; self.pc = self.pila.pop() if self.pila else 0; self.ciclos += 1
        elif op & 0x3F00 == 0x3800: self.w |= k; self.flags(z=self.w == 0)
        elif op & 0x3F00 == 0x3900: self.w &= k; self.flags(z=self.w == 0)
        elif op & 0x3F00 == 0x3A00: self.w ^= k; self.flags(z=self.w == 0)
        elif op & 0x3E00 == 0x3C00:
            r = k - self.w; self.flags(c=r >= 0, dc=(k & 15) >= (self.w & 15)); self.w = r & 0xFF; self.flags(z=self.w == 0)
        elif op & 0x3E00 == 0x3E00:
            r = k + self.w; self.flags(c=r > 255, dc=((k & 15) + (self.w & 15)) > 15); self.w = r & 0xFF; self.flags(z=self.w == 0)
        else: raise ValueError(f'opcode {op:04X} en {pc0:04X}')

    def correr(self, ciclos):
        fin = self.ciclos + ciclos
        while self.ciclos < fin:
            self.paso(); self.perifericos()

    def puerto(self, p):
        a = {'A': 5, 'B': 6, 'C': 7, 'D': 8, 'E': 9}[p]
        return self.leer(a)

## 2. Los cuatro programas de ejemplo
Cada ejemplo está en ensamblador (es el que se convierte en `.hex`) y en C para XC8 (para compilar en MPLAB X).

### Ejemplo 1 · secuencia de LEDs en PORTB con retardo de 100 ms

In [ ]:
%%writefile ej1_secuencia_leds.asm
;=====================================================================
; Ejemplo 1 · Secuencia de LEDs «ida y vuelta» en PORTB
; PIC16F877A · cristal de 4 MHz (ciclo de instrucción = 1 µs)
; LEDs en RB0..RB7 con resistencias de 330 Ω a tierra
;=====================================================================
        LIST    p=16F877A
        #include <p16f877a.inc>
        __CONFIG 0x3F31         ; XT, WDT off, PWRT on, BOR off, LVP off, sin protección

        CBLOCK  0x20
        d1                      ; contador del lazo interno
        d3                      ; contador de milisegundos
        ENDC

        ORG     0x0000
        goto    inicio

inicio  banksel TRISB
        clrf    TRISB           ; PORTB: las 8 líneas como salidas
        banksel PORTB
        movlw   b'00000001'
        movwf   PORTB           ; enciende RB0

izq     call    retardo         ; desplaza hacia RB7
        bcf     STATUS,C
        rlf     PORTB,F
        btfss   PORTB,7
        goto    izq
der     call    retardo         ; desplaza hacia RB0
        bcf     STATUS,C
        rrf     PORTB,F
        btfss   PORTB,0
        goto    der
        goto    izq

;--- retardo de 100 ms: 100 × 1 ms ----------------------------------
retardo movlw   d'100'
        movwf   d3
r1      movlw   d'249'          ; 249 × 4 ciclos ≈ 1 ms
        movwf   d1
r2      nop
        decfsz  d1,F
        goto    r2
        decfsz  d3,F
        goto    r1
        return

        END

In [ ]:
%%writefile ej1_secuencia_leds.c
/*====================================================================
 * Ejemplo 1 · Secuencia de LEDs «ida y vuelta» en PORTB
 * PIC16F877A · compilador MPLAB XC8 · cristal de 4 MHz
 *====================================================================*/
#include <xc.h>
#include <stdint.h>

// Bits de configuración (PIC16F877A). En MPLAB X: Production > Set Configuration Bits
// genera estas mismas líneas para el PIC exacto que use (el PIC16F877 tiene opciones casi iguales).
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL     // cristal de 4 MHz, necesario para __delay_ms()

void main(void) {
    uint8_t led = 0x01;
    TRISB = 0x00;                       // PORTB: salidas
    while (1) {
        while (led != 0x80) {           // hacia RB7
            PORTB = led;
            __delay_ms(100);
            led <<= 1;
        }
        while (led != 0x01) {           // hacia RB0
            PORTB = led;
            __delay_ms(100);
            led >>= 1;
        }
    }
}

### Ejemplo 2 · contador 0–9 en display de 7 segmentos con pulsador en RB0

In [ ]:
%%writefile ej2_contador_7seg.asm
;=====================================================================
; Ejemplo 2 · Contador 0–9 en display de 7 segmentos con pulsador
; PIC16F877A · 4 MHz · display cátodo común: a..g = RD0..RD6
; Pulsador en RB0 con resistencia de 10 kΩ a +5 V (activo en bajo)
;=====================================================================
        LIST    p=16F877A
        #include <p16f877a.inc>
        __CONFIG 0x3F31

        CBLOCK  0x20
        cuenta
        d1
        d3
        ENDC

        ORG     0x0000
        goto    inicio

;--- tabla de segmentos (gfedcba) ------------------------------------
tabla   addwf   PCL,F
        DT      0x3F,0x06,0x5B,0x4F,0x66,0x6D,0x7D,0x07,0x7F,0x6F

inicio  banksel TRISD
        clrf    TRISD           ; PORTD: segmentos
        movlw   0xFF
        movwf   TRISB           ; PORTB: entradas
        banksel PORTD
        clrf    cuenta

mostrar movf    cuenta,W
        call    tabla
        movwf   PORTD
esperaP btfsc   PORTB,0         ; espera la pulsación (RB0 = 0)
        goto    esperaP
        call    rebote
esperaS btfss   PORTB,0         ; espera que se suelte
        goto    esperaS
        call    rebote
        incf    cuenta,F        ; cuenta = cuenta + 1
        movf    cuenta,W
        xorlw   d'10'           ; ¿llegó a 10?
        btfsc   STATUS,Z
        clrf    cuenta          ; sí: vuelve a 0
        goto    mostrar

;--- antirrebote de 20 ms ---------------------------------------------
rebote  movlw   d'20'
        movwf   d3
r1      movlw   d'249'
        movwf   d1
r2      nop
        decfsz  d1,F
        goto    r2
        decfsz  d3,F
        goto    r1
        return

        END

In [ ]:
%%writefile ej2_contador_7seg.c
/*====================================================================
 * Ejemplo 2 · Contador 0–9 en display de 7 segmentos con pulsador
 * PIC16F877A · XC8 · 4 MHz · display cátodo común a..g = RD0..RD6
 * Pulsador en RB0 con resistencia de 10 kΩ a +5 V (activo en bajo)
 *====================================================================*/
#include <xc.h>
#include <stdint.h>

// Bits de configuración (PIC16F877A). En MPLAB X: Production > Set Configuration Bits
// genera estas mismas líneas para el PIC exacto que use (el PIC16F877 tiene opciones casi iguales).
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL     // cristal de 4 MHz, necesario para __delay_ms()

const uint8_t SEGMENTOS[10] = {0x3F, 0x06, 0x5B, 0x4F, 0x66, 0x6D, 0x7D, 0x07, 0x7F, 0x6F};

void main(void) {
    uint8_t cuenta = 0;
    TRISD = 0x00;                       // segmentos
    TRISB = 0xFF;                       // pulsador
    while (1) {
        PORTD = SEGMENTOS[cuenta];
        while (PORTBbits.RB0) { }       // espera la pulsación
        __delay_ms(20);                 // antirrebote
        while (!PORTBbits.RB0) { }      // espera que se suelte
        __delay_ms(20);
        if (++cuenta == 10) cuenta = 0;
    }
}

### Ejemplo 3 · potenciómetro en AN0 → ADC → PWM en RC2/CCP1

In [ ]:
%%writefile ej3_adc_pwm.asm
;=====================================================================
; Ejemplo 3 · Potenciómetro (AN0) → PWM en RC2/CCP1 y valor en PORTB
; PIC16F877A · 4 MHz · PWM de 976,6 Hz (PR2 = 255, preescalador 4)
; ADC justificado a la izquierda: ADRESH = 8 bits más significativos
;=====================================================================
        LIST    p=16F877A
        #include <p16f877a.inc>
        __CONFIG 0x3F31

        CBLOCK  0x20
        d1
        ENDC

        ORG     0x0000
        goto    inicio

inicio  banksel TRISA
        movlw   0xFF
        movwf   TRISA           ; PORTA: entradas (AN0)
        clrf    TRISB           ; PORTB: LEDs con el valor del ADC
        bcf     TRISC,2         ; RC2/CCP1: salida PWM
        movlw   b'00001110'     ; ADFM = 0 (izquierda), PCFG = 1110: solo AN0 analógico
        movwf   ADCON1
        movlw   d'255'
        movwf   PR2             ; periodo PWM = (255+1)·4·Tosc·4 = 1,024 ms
        banksel ADCON0
        movlw   b'01000001'     ; Fosc/8 (Tad = 2 µs), canal 0, ADON = 1
        movwf   ADCON0
        movlw   b'00000101'     ; TMR2 encendido, preescalador 1:4
        movwf   T2CON
        movlw   b'00001100'     ; CCP1 en modo PWM
        movwf   CCP1CON
        clrf    CCPR1L

bucle   movlw   d'7'            ; tiempo de adquisición ≈ 20 µs
        movwf   d1
adq     decfsz  d1,F
        goto    adq
        bsf     ADCON0,GO       ; inicia la conversión
espera  btfsc   ADCON0,GO       ; espera GO/DONE = 0
        goto    espera
        movf    ADRESH,W
        movwf   CCPR1L          ; ciclo útil = ADRESH/256
        movwf   PORTB           ; muestra el valor en los LEDs
        goto    bucle

        END

In [ ]:
%%writefile ej3_adc_pwm.c
/*====================================================================
 * Ejemplo 3 · Potenciómetro (AN0) → PWM en RC2/CCP1 y valor en PORTB
 * PIC16F877A · XC8 · 4 MHz · PWM de 976,6 Hz (PR2 = 255, preescalador 4)
 *====================================================================*/
#include <xc.h>
#include <stdint.h>

// Bits de configuración (PIC16F877A). En MPLAB X: Production > Set Configuration Bits
// genera estas mismas líneas para el PIC exacto que use (el PIC16F877 tiene opciones casi iguales).
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL     // cristal de 4 MHz, necesario para __delay_ms()

void main(void) {
    TRISA = 0xFF;                       // AN0 entrada
    TRISB = 0x00;                       // LEDs
    TRISCbits.TRISC2 = 0;               // salida PWM
    ADCON1 = 0x0E;                      // justificado a la izquierda, solo AN0 analógico
    ADCON0 = 0x41;                      // Fosc/8 (Tad = 2 µs), canal 0, ADON = 1
    PR2 = 255;                          // periodo = (PR2 + 1)·4·Tosc·4 = 1,024 ms
    T2CON = 0x05;                       // TMR2 encendido, preescalador 1:4
    CCP1CON = 0x0C;                     // CCP1 en modo PWM
    CCPR1L = 0;
    while (1) {
        __delay_us(20);                 // tiempo de adquisición
        ADCON0bits.GO_nDONE = 1;        // inicia la conversión
        while (ADCON0bits.GO_nDONE) { } // espera el final
        CCPR1L = ADRESH;                // ciclo útil = ADRESH / 256
        PORTB = ADRESH;
    }
}

### Ejemplo 4 · USART a 9600 baudios con eco

In [ ]:
%%writefile ej4_usart.asm
;=====================================================================
; Ejemplo 4 · USART a 9600 baudios: mensaje de bienvenida y eco
; PIC16F877A · 4 MHz · SPBRG = 25 con BRGH = 1 → 9615 baudios (0,16 %)
; TX = RC6, RX = RC7 · el último carácter recibido se ve en PORTB
;=====================================================================
        LIST    p=16F877A
        #include <p16f877a.inc>
        __CONFIG 0x3F31

        CBLOCK  0x20
        indice
        ENDC

        ORG     0x0000
        goto    inicio

;--- mensaje terminado en 0 -------------------------------------------
mensaje addwf   PCL,F
        DT      "PIC16F877A listo", 0x0D, 0x0A, 0

inicio  banksel TRISC
        bcf     TRISC,6         ; TX salida
        bsf     TRISC,7         ; RX entrada
        clrf    TRISB
        movlw   d'25'
        movwf   SPBRG           ; 4 MHz / (16 · 26) = 9615 baudios
        movlw   b'00100100'     ; TXEN = 1, BRGH = 1, asíncrono
        movwf   TXSTA
        banksel RCSTA
        movlw   b'10010000'     ; SPEN = 1, CREN = 1
        movwf   RCSTA
        clrf    PORTB

        clrf    indice
enviar  movf    indice,W
        call    mensaje
        iorlw   0               ; ¿fin del texto?
        btfsc   STATUS,Z
        goto    eco
        call    tx
        incf    indice,F
        goto    enviar

eco     btfss   PIR1,RCIF       ; espera un carácter
        goto    eco
        movf    RCREG,W
        movwf   PORTB           ; lo muestra en los LEDs
        call    tx              ; y lo devuelve
        goto    eco

tx      btfss   PIR1,TXIF       ; espera que TXREG esté libre
        goto    tx
        movwf   TXREG
        return

        END

In [ ]:
%%writefile ej4_usart.c
/*====================================================================
 * Ejemplo 4 · USART a 9600 baudios: mensaje de bienvenida y eco
 * PIC16F877A · XC8 · 4 MHz · SPBRG = 25 con BRGH = 1 → 9615 baudios
 *====================================================================*/
#include <xc.h>
#include <stdint.h>

// Bits de configuración (PIC16F877A). En MPLAB X: Production > Set Configuration Bits
// genera estas mismas líneas para el PIC exacto que use (el PIC16F877 tiene opciones casi iguales).
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL     // cristal de 4 MHz, necesario para __delay_ms()

void uart_tx(char c) {
    while (!PIR1bits.TXIF) { }          // espera a que TXREG esté libre
    TXREG = c;
}

void uart_texto(const char *s) {
    while (*s) uart_tx(*s++);
}

void main(void) {
    TRISCbits.TRISC6 = 0;               // TX
    TRISCbits.TRISC7 = 1;               // RX
    TRISB = 0x00;
    SPBRG = 25;                         // 4 MHz / (16 · (25 + 1)) = 9615 baudios
    TXSTA = 0x24;                       // TXEN = 1, BRGH = 1, asíncrono
    RCSTA = 0x90;                       // SPEN = 1, CREN = 1
    PORTB = 0;
    uart_texto("PIC16F877A listo\r\n");
    while (1) {
        if (RCSTAbits.OERR) {           // desbordamiento: reinicia el receptor
            RCSTAbits.CREN = 0;
            RCSTAbits.CREN = 1;
        }
        if (PIR1bits.RCIF) {
            char c = RCREG;
            PORTB = c;                  // muestra el carácter en los LEDs
            uart_tx(c);                 // eco
        }
    }
}

## 3. Ensamblar y generar los `.hex`
La celda crea `ejN_*.hex` y `ejN_*.lst` (listado con dirección, código de máquina y mnemónico). En Proteus: doble clic sobre el PIC → *Program File* = el `.hex`, *Processor Clock Frequency* = 4 MHz.

In [ ]:
import importlib, picasm, pic16sim
importlib.reload(picasm); importlib.reload(pic16sim)
from picasm import Ensamblador, leer_hex, desensamblar
from pic16sim import PIC16F877

EJEMPLOS = ['ej1_secuencia_leds', 'ej2_contador_7seg', 'ej3_adc_pwm', 'ej4_usart']
PROG = {}
for n in EJEMPLOS:
    a = Ensamblador(open(n + '.asm', encoding='utf-8').read()).ensamblar()
    open(n + '.hex', 'w').write(a.hex())
    with open(n + '.lst', 'w', encoding='utf-8') as L:
        for pc, w, t in a.lista: L.write(f'{pc:04X}  {w:04X}  {desensamblar(w):<22s} ; {t}\n')
    PROG[n], conf = leer_hex(a.hex())
    print(f'{n + ".hex":26s} {len(PROG[n]):3d} palabras de programa · palabra de configuración 0x{conf:04X}')

print('\nContenido de ej1_secuencia_leds.hex (formato Intel HEX):')
print(open('ej1_secuencia_leds.hex').read())

In [ ]:
# Listado: dirección, código de máquina de 14 bits y mnemónico
print(open('ej1_secuencia_leds.lst', encoding='utf-8').read())

In [ ]:
# Descargar todos los .hex, .asm, .c y .lst en un .zip (solo en Colab)
import zipfile, glob
with zipfile.ZipFile('ejemplos_pic16f877a.zip', 'w') as z:
    for f in sorted(glob.glob('ej[1-4]_*.*')):
        if not f.endswith('.zip'): z.write(f)
print('Creado ejemplos_pic16f877a.zip')
try:
    from google.colab import files
    files.download('ejemplos_pic16f877a.zip')
except ImportError:
    print('(Fuera de Colab: el archivo queda en la carpeta actual.)')

## 4. Arquitectura: del código en C al código de máquina

Las 35 instrucciones del PIC16 caben en **tres formatos** de 14 bits:

| Formato | Bits | Ejemplo |
|---|---|---|
| Orientado a byte | `00 oooo d fffffff` | `addwf PCL,F` |
| Orientado a bit | `01 oo bbb fffffff` | `bsf STATUS,RP0` |
| Literal / salto | `11 oooo kkkkkkkk` · `10 o kkkkkkkkkkk` | `movlw 0xFF` · `goto 0x00C` |

La siguiente celda ensambla instrucciones sueltas y separa sus campos.

In [ ]:
def campos(w):
    b = f'{w:014b}'
    if w >> 12 == 0:   return f'{b[:2]} {b[2:6]} {b[6]} {b[7:]}   (byte: op, d, f)'
    if w >> 12 == 1:   return f'{b[:2]} {b[2:4]} {b[4:7]} {b[7:]}   (bit: op, b, f)'
    if w >> 12 == 2:   return f'{b[:3]} {b[3:]}   (call/goto: op, k de 11 bits)'
    return f'{b[:6]} {b[6:]}   (literal: op, k de 8 bits)'

for src in ['movlw 0x55', 'movwf PORTB', 'bsf STATUS,RP0', 'clrf TRISB', 'addwf PCL,F', 'decfsz 0x20,F',
            'btfss PIR1,TXIF', 'goto 0x00C', 'call 0x123', 'retlw 0x3F', 'return']:
    w = Ensamblador('        ' + src + '\n').ensamblar().mem[0]
    print(f'{src:18s} → 0x{w:04X} = {campos(w)}')

In [ ]:
# Ejercicio: decodifique a mano estas palabras y compruebe con el desensamblador
for w in [0x3007, 0x0086, 0x1386, 0x0B8D, 0x2805]:
    print(f'0x{w:04X} → {desensamblar(w)}')

### 4.1 Von Neumann frente a Harvard
En **Von Neumann** las instrucciones y los datos comparten un solo bus: una instrucción que lee un dato necesita dos accesos seguidos. En **Harvard** (PIC, AVR) hay dos buses, y la búsqueda de la siguiente instrucción se hace en paralelo con el acceso al dato.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

rng = np.random.default_rng(1)
N = 40
usa_dato = rng.random(N) < 0.6          # 60 % de las instrucciones leen o escriben un dato
vn = np.cumsum(1 + usa_dato)            # 1 acceso para la instrucción + 1 por el dato (mismo bus)
hv = np.arange(1, N + 1)                # buses separados: un acceso por instrucción
plt.figure(figsize=(8, 3.6))
plt.step(np.arange(1, N + 1), vn, where='post', label='Von Neumann (un bus)', lw=2.2, color='#D2402F')
plt.step(np.arange(1, N + 1), hv, where='post', label='Harvard (dos buses)', lw=2.2, color='#1F8A70')
plt.xlabel('instrucciones ejecutadas'); plt.ylabel('ciclos de bus'); plt.legend(frameon=False)
plt.title('Ciclos de bus acumulados'); plt.show()
print(f'Von Neumann: {vn[-1]} ciclos · Harvard: {hv[-1]} ciclos · aceleración ≈ {vn[-1]/hv[-1]:.2f}×')

### 4.2 CISC frente a RISC
Tiempo de ejecución = instrucciones × CPI × periodo de reloj. Un **CISC** usa menos instrucciones pero cada una tarda varios ciclos; un **RISC** usa más instrucciones, casi todas de un ciclo. Ejemplo: sumar dos variables en memoria y guardar el resultado.

In [ ]:
casos = {
    'CISC (8051, 12 relojes/ciclo)': dict(instr=3, cpi=1.3, fclk=12e6, reloj_por_ciclo=12),
    'RISC PIC16 (4 relojes/ciclo)':  dict(instr=3, cpi=1.0, fclk=20e6, reloj_por_ciclo=4),
    'RISC AVR (1 reloj/ciclo)':      dict(instr=4, cpi=1.0, fclk=20e6, reloj_por_ciclo=1),
}
nombres, tiempos = [], []
for n, c in casos.items():
    t = c['instr'] * c['cpi'] * c['reloj_por_ciclo'] / c['fclk']
    nombres.append(n); tiempos.append(t * 1e6)
    mips = c['fclk'] / c['reloj_por_ciclo'] / c['cpi'] / 1e6
    print(f'{n:32s} {c["instr"]} instr · {mips:5.2f} MIPS · {t*1e6:6.3f} µs')
plt.figure(figsize=(8, 2.8))
plt.barh(nombres, tiempos, color=['#7B5EA7', '#1F8A70', '#2F6DB5']); plt.xlabel('tiempo (µs)')
plt.title('a = b + c  en tres arquitecturas'); plt.gca().invert_yaxis(); plt.show()

### 4.3 Segmentación (pipeline) de dos etapas del PIC16
Mientras se **ejecuta** una instrucción, ya se **busca** la siguiente. Por eso casi todas tardan 1 ciclo; los saltos (`goto`, `call`, `return`, saltos condicionales tomados) tardan 2 porque hay que descartar la instrucción ya buscada.

In [ ]:
prog = ['movlw 0x55', 'movwf PORTB', 'goto LAZO', 'nop (descartada)', 'LAZO: incf PORTB,F']
fig, ax = plt.subplots(figsize=(8.5, 2.8))
eventos = [(0, 0, 'B'), (0, 1, 'E'), (1, 1, 'B'), (1, 2, 'E'), (2, 2, 'B'), (2, 3, 'E'), (3, 3, 'B'), (3, 4, 'descartada'), (4, 4, 'B'), (4, 5, 'E')]
for i, t, tipo in eventos:
    col = {'B': '#2F6DB5', 'E': '#1F8A70', 'descartada': '#C9D2D6'}[tipo]
    ax.barh(i, 1, left=t, color=col, edgecolor='white'); ax.text(t + 0.5, i, {'B': 'búsqueda', 'E': 'ejecución'}.get(tipo, 'vacía'), ha='center', va='center', color='white' if tipo != 'descartada' else '#333', fontsize=9)
ax.set_yticks(range(len(prog))); ax.set_yticklabels(prog); ax.invert_yaxis(); ax.set_xlabel('ciclo de instrucción (Tcy = 1 µs a 4 MHz)')
ax.set_title('Pipeline de 2 etapas: goto ocupa 2 ciclos'); plt.show()

## 5. Cálculos de diseño

In [ ]:
FOSC = 4e6
TCY = 4 / FOSC
print(f'Ciclo de instrucción Tcy = 4/Fosc = {TCY*1e6:.1f} µs → {1/TCY/1e6:.0f} MIPS')

# Retardo con dos lazos anidados (ejemplo 1): 100 × (249 × 4 + 4) ciclos
interno, externo = 249, 100
ciclos = externo * (interno * 4 + 4)
print(f'Retardo = {externo} × ({interno}×4 + 4) = {ciclos} ciclos = {ciclos*TCY*1e3:.2f} ms')

# Resistencia de los LED
VDD, VLED, I = 5.0, 2.0, 9e-3
R = (VDD - VLED) / I
print(f'R_LED = (5 − 2) V / 9 mA = {R:.0f} Ω → comercial 330 Ω → I = {(VDD-VLED)/330*1e3:.1f} mA por LED; 8 LED = {8*(VDD-VLED)/330*1e3:.0f} mA (límite 200 mA por grupo de puertos)')

In [ ]:
# ADC de 10 bits
VREF = 5.0
paso = VREF / 1024
TAD = 8 / FOSC
print(f'Resolución = 5 V / 1024 = {paso*1e3:.2f} mV · TAD = 8/Fosc = {TAD*1e6:.0f} µs · conversión = 12 TAD = {12*TAD*1e6:.0f} µs')
v = np.linspace(0, 5, 500)
cod = np.clip(np.round(v / VREF * 1023), 0, 1023)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
ax[0].plot(v, cod, color='#1F8A70'); ax[0].set_xlabel('V en AN0 (V)'); ax[0].set_ylabel('ADRESH:ADRESL'); ax[0].set_title('Función de transferencia (10 bits)')
z = np.linspace(0, 0.05, 400); ax[1].step(z * 1e3, np.floor(z / paso), where='post', color='#F2A33A'); ax[1].set_xlabel('mV'); ax[1].set_ylabel('código'); ax[1].set_title('Zoom: escalones de 4,88 mV')
plt.tight_layout(); plt.show()
for vin in (0.0, 1.25, 2.5, 3.75, 5.0):
    r = round(vin / 5 * 1023); print(f'{vin:4.2f} V → código {r:4d} → ADRESH (justificado a la izquierda) = {r >> 2:3d}')

In [ ]:
# PWM del CCP1: periodo = (PR2 + 1)·4·Tosc·prescaler ;  ciclo útil = CCPR1L:CCP1CON<5:4> · Tosc · prescaler
def pwm(pr2, pres, ccpr, fosc=FOSC):
    T = (pr2 + 1) * 4 * pres / fosc
    return 1 / T, min(1.0, (ccpr << 2) * pres / fosc / T)
for pres in (1, 4, 16):
    f, _ = pwm(255, pres, 0); print(f'PR2 = 255, prescaler {pres:2d} → f = {f:8.1f} Hz')
f, D = pwm(255, 4, 128); print(f'Ejemplo 3: PR2 = 255, prescaler 4, CCPR1L = 128 → f = {f:.1f} Hz, D = {D*100:.1f} %')
pr = np.arange(1, 256)
plt.figure(figsize=(8, 3))
for pres, c in zip((1, 4, 16), ('#2F6DB5', '#1F8A70', '#7B5EA7')):
    plt.semilogy(pr, [pwm(p, pres, 0)[0] for p in pr], label=f'prescaler {pres}', color=c)
plt.xlabel('PR2'); plt.ylabel('f PWM (Hz)'); plt.legend(frameon=False); plt.title('Frecuencia del PWM con cristal de 4 MHz'); plt.show()

In [ ]:
# USART: baudios = Fosc / (16·(SPBRG+1)) con BRGH = 1
print(' baudios  SPBRG  real      error')
for b in (1200, 2400, 4800, 9600, 19200):
    for brgh, div in ((1, 16), (0, 64)):
        s = round(FOSC / (div * b) - 1)
        if 0 <= s <= 255: break
    real = FOSC / (div * (s + 1))
    print(f'{b:7d}  {s:5d}  {real:8.1f}  {abs(real-b)/b*100:5.2f} %  (BRGH={brgh})')

## 6. Simulación de los ejemplos
El emulador ejecuta exactamente los `.hex` generados arriba (los mismos de Proteus).

In [ ]:
# Ejemplo 1: PORTB en el tiempo (cada fila es un bit; amarillo = LED encendido)
p = PIC16F877(PROG['ej1_secuencia_leds'])
muestras = []
for k in range(40):
    p.correr(50_000); muestras.append([(p.puerto('B') >> b) & 1 for b in range(8)])
M = np.array(muestras).T
plt.figure(figsize=(9, 3)); plt.imshow(M, aspect='auto', cmap='cividis', origin='lower', extent=[0, 40 * 50, -0.5, 7.5])
plt.yticks(range(8), [f'RB{b}' for b in range(8)]); plt.xlabel('tiempo (ms)'); plt.title('Ejemplo 1: la luz va y vuelve cada 100 ms'); plt.show()

# medir con el emulador el tiempo entre cambios
p = PIC16F877(PROG['ej1_secuencia_leds']); ult = None; t0 = 0; difs = []
while len(difs) < 4:
    p.paso(); v = p.ram[6]
    if v != ult:
        if ult is not None: difs.append(p.ciclos - t0)
        ult, t0 = v, p.ciclos
print('Ciclos entre cambios:', difs[1:], '→', difs[-1] * TCY * 1e3, 'ms')

In [ ]:
# Ejemplo 2: cada pulsación en RB0 incrementa el display
SEG = {0x3F: 0, 0x06: 1, 0x5B: 2, 0x4F: 3, 0x66: 4, 0x6D: 5, 0x7D: 6, 0x07: 7, 0x7F: 8, 0x6F: 9}
def dibujar7(ax, cod):
    s = {'a': [(0, 2), (1, 2)], 'b': [(1, 2), (1, 1)], 'c': [(1, 1), (1, 0)], 'd': [(0, 0), (1, 0)], 'e': [(0, 1), (0, 0)], 'f': [(0, 2), (0, 1)], 'g': [(0, 1), (1, 1)]}
    for i, k in enumerate('abcdefg'):
        (x1, y1), (x2, y2) = s[k]; ax.plot([x1, x2], [y1, y2], lw=6, solid_capstyle='round', color='#FF4A3A' if (cod >> i) & 1 else '#EEDDDD')
    ax.set_xlim(-0.4, 1.4); ax.set_ylim(-0.3, 2.3); ax.axis('off'); ax.set_aspect('equal')

p = PIC16F877(PROG['ej2_contador_7seg']); p.entradas['B'] = 0xFF; p.correr(5000)
codigos = [p.puerto('D')]
for k in range(11):
    p.entradas['B'] = 0xFE; p.correr(40_000)    # pulsador presionado 40 ms
    p.entradas['B'] = 0xFF; p.correr(40_000)    # suelto
    codigos.append(p.puerto('D'))
fig, axs = plt.subplots(1, len(codigos), figsize=(12, 1.8))
for ax, c in zip(axs, codigos): dibujar7(ax, c); ax.set_title(f'0x{c:02X}', fontsize=8)
plt.suptitle('Ejemplo 2: PORTD después de cada pulsación'); plt.show()
print('Dígitos:', [SEG.get(c, '?') for c in codigos])

In [ ]:
# Ejemplo 3: barrido del potenciómetro y forma de onda en RC2
p = PIC16F877(PROG['ej3_adc_pwm'])
vs = np.linspace(0, 5, 21); Ds = []; adr = []
for v in vs:
    p.analog[0] = v; p.correr(3000); Ds.append(p.pwm(1)['D'] * 100); adr.append(p.ram[0x1E])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(vs, Ds, 'o-', color='#1F8A70'); ax[0].set_xlabel('V en AN0 (V)'); ax[0].set_ylabel('ciclo útil (%)'); ax[0].set_title('ADC → PWM')
for v, c in ((1.25, '#2F6DB5'), (2.5, '#1F8A70'), (3.75, '#D2402F')):
    p.analog[0] = v; p.correr(3000); r = p.pwm(1)
    t = np.linspace(0, 3 / r['f'], 1500); ax[1].plot(t * 1e3, ((t * r['f']) % 1 < r['D']) * 5 + (v - 2.5) * 0.04, color=c, label=f'{v} V → {r["D"]*100:.0f} %')
ax[1].set_xlabel('ms'); ax[1].set_ylabel('RC2 (V)'); ax[1].legend(frameon=False, fontsize=8); ax[1].set_title(f'PWM a {r["f"]:.1f} Hz')
plt.tight_layout(); plt.show()

In [ ]:
# Ejemplo 4: mensaje inicial y eco
p = PIC16F877(PROG['ej4_usart']); p.correr(30_000)
print('El PIC transmite:', repr(bytes(p.tx).decode()))
for ch in b'Hola PIC':
    p.recibir(ch); p.correr(2000)
print('Después del eco :', repr(bytes(p.tx).decode()))
print(f'Tiempo por carácter = {p.ciclos_char()} µs (10 bits a 9615 baudios)')

## 7. Ejercicios propuestos
1. **Retardo**: cambie el ejemplo 1 para que la secuencia avance cada 250 ms. ¿Qué valores de los contadores usa? Verifíquelo midiendo ciclos con el emulador.
2. **Contador descendente**: modifique el ejemplo 2 para que cuente de 9 a 0 (pista: `decf` y compare con 0xFF).
3. **Umbral**: en el ejemplo 3, encienda RB7 solo si el voltaje del potenciómetro es mayor que 3 V (ADRESH > 153).
4. **Comando por la USART**: en el ejemplo 4, haga que la letra `A` encienda todos los LED y `B` los apague.

Plantilla: escriba su programa en la celda siguiente, ejecútela y observe PORTB.

In [ ]:
MI_PROGRAMA = """
        LIST p=16F877A
        #include <p16f877a.inc>
        __CONFIG 0x3F31
        ORG 0x000
        BANKSEL TRISB
        clrf    TRISB
        BANKSEL PORTB
        movlw   0xA5
        movwf   PORTB
fin     goto    fin
        END
"""
a = Ensamblador(MI_PROGRAMA).ensamblar()
open('mi_programa.hex', 'w').write(a.hex())
p = PIC16F877(a.mem); p.correr(1000)
print('PORTB =', f'{p.puerto("B"):08b}', '· se guardó mi_programa.hex para Proteus')

## 8. Simulador interactivo en el navegador
Suba el archivo `simulador_pic16f877.html` (entregado con la presentación) y ejecute la celda: la página se muestra aquí mismo, con desplazamiento propio.

In [ ]:
import os, html as _html
from IPython.display import HTML, display
f = 'simulador_pic16f877.html'
if not os.path.exists(f):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print('Copie simulador_pic16f877.html a esta carpeta.')
if os.path.exists(f):
    PAGINA = open(f, encoding='utf-8').read()
    display(HTML('<iframe srcdoc="' + _html.escape(PAGINA, quote=True) + '" style="width:100%;height:900px;border:1px solid #D5DEE2;border-radius:12px" scrolling="yes"></iframe>'))